# 📊 ECE Lab Kit Sales — Master Dashboard

Season / Course / Item modes are **independent**. Advanced panels collapse/expand via the grey buttons. Run cells **1 → 6** in order.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted.')

In [ ]:
# IMPORTANT: must run before the dashboard cell
from google.colab import output
output.enable_custom_widget_manager()
print('✅ Custom widget manager enabled.')

In [ ]:
SHEET_ID   = 'YOUR_GOOGLE_SHEET_ID_HERE'
SHEET_NAME = 'Sheet3'

EXPORT_URL=f'https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={SHEET_NAME}'
print(f'URL: {EXPORT_URL}')

In [ ]:
import pandas as pd,numpy as np,requests
from io import StringIO

try:
    resp=requests.get(EXPORT_URL,timeout=15); resp.raise_for_status()
    raw=pd.read_csv(StringIO(resp.text),header=None); print('✅ Loaded from Sheets.')
except Exception as e:
    print(f'Fallback: {e}')
    raw=pd.read_csv('/content/drive/MyDrive/Summer_2026_Sale_-_Sheet3.csv',header=None)
    print('✅ Loaded from Drive.')
raw.head()

In [ ]:
semester_labels=raw.iloc[0,1:].tolist()
COURSES_CLASS=['ECE 3741','ECE 3043','ECE 2040','ECE 2031','ECE 2035']
ITEM_KEYS=['Giant Br','Small Br','Wirekits','Credit Card','# Sales']

class_size_rows,kits_course_rows,rate_rows,item_rows={},{},{},{}
in_kits=False;in_rate=False

for _,row in raw.iterrows():
    label=str(row.iloc[0]).strip()
    if not label or label=='nan':continue
    if label=='Kits Purchased':in_kits,in_rate=True,False;continue
    if label in('Purcahse Rate','Purchase Rate'):in_rate,in_kits=True,False;continue
    vals=pd.to_numeric(row.iloc[1:],errors='coerce');vals.index=semester_labels
    if not in_kits and not in_rate and label in COURSES_CLASS:class_size_rows[label]=vals
    elif in_kits and label in COURSES_CLASS:kits_course_rows[label]=vals
    elif in_kits and label.strip() in ITEM_KEYS:item_rows[label.strip()]=vals
    elif in_rate and label in['ECE 3741','ECE 3043','ECE 2031']:rate_rows[label]=vals

ch=lambda df:df.iloc[:,::-1];chs=lambda s:s.iloc[::-1]
class_size_df=ch(pd.DataFrame(class_size_rows).T)
kits_course_df=ch(pd.DataFrame(kits_course_rows).T)
rate_df=ch(pd.DataFrame(rate_rows).T)
item_series={k:chs(v) for k,v in item_rows.items()}
sales_series=item_series.get('# Sales',pd.Series(dtype=float))
for df in[class_size_df,kits_course_df,rate_df]:
    if 'Fall 2026' in df.columns:df.drop(columns=['Fall 2026'],inplace=True)
item_series={k:s.drop('Fall 2026',errors='ignore') for k,s in item_series.items()}
sales_series=sales_series.drop('Fall 2026',errors='ignore')
print('Parsed:',class_size_df.shape,kits_course_df.shape,sorted(item_series))
print('✅ Done — run Cell 6.')

In [ ]:
# ── Cell 6: Master Interactive Dashboard ──────────────────────────────────────
import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display
from scipy import stats as spstats
import numpy as np
import pandas as pd
import warnings; warnings.filterwarnings('ignore')

# ── Constants ─────────────────────────────────────────────────────────────────
PALETTE = ['#003057','#E87722','#B3A369','#76b900',
           '#984ea3','#e41a1c','#54585A','#ff7f00','#1f78b4','#33a02c']
SEASON_ORDER  = ['Spring','Summer','Fall']
SEASON_COLORS = {'Spring':'#2166ac','Summer':'#E87722','Fall':'#003057'}
ITEMS_LIST    = ['Giant Br','Small Br','Wirekits','Credit Card','# Sales']
COURSE_METRICS = {'class_size','kits_by_course','purchase_rate'}
METRIC_YLABEL = {
    'class_size':'Students Enrolled','kits_by_course':'Kits Sold',
    'purchase_rate':'Purchase Rate (%)','giant_br':'Giant Breadboards Sold',
    'small_br':'Small Breadboards Sold','wirekits':'Wire Kits Sold',
    'credit_card':'Credit Card Transactions','sales':'Total Sales',
    'item_aggregate':'Aggregated Items',
}
METRIC_OPTIONS = [
    ('Class Size (by course)','class_size'),('Kits Sold (by course)','kits_by_course'),
    ('Purchase Rate (by course)','purchase_rate'),('Giant Breadboards','giant_br'),
    ('Small Breadboards','small_br'),('Wire Kits','wirekits'),
    ('Credit Card Charges','credit_card'),('# Total Sales','sales'),
    ('Custom Item Aggregate','item_aggregate'),
]
TRANSFORM_OPTIONS = ['Raw','YoY Growth (%)','YoY Difference','Cumulative']

# ── Data helpers ───────────────────────────────────────────────────────────────
def get_checked(d): return [k for k,cb in d.items() if cb.value]

def get_season_data(obj, season):
    """Single season → re-index by year (no aggregation needed)."""
    if isinstance(obj, pd.Series):
        kept=[i for i in obj.index if str(i).startswith(season)]
        if not kept: return pd.Series(dtype=float)
        s=obj[kept].copy(); s.index=[i.split()[-1] for i in s.index]
        return s.sort_index()
    cols=[c for c in obj.columns if str(c).startswith(season)]
    if not cols: return pd.DataFrame(dtype=float)
    sub=obj[cols].copy(); sub.columns=[c.split()[-1] for c in sub.columns]
    return sub.sort_index(axis=1)

def aggregate_by_year(obj, seasons, is_rate=False):
    if not seasons:
        return pd.Series(dtype=float) if isinstance(obj,pd.Series) else pd.DataFrame(dtype=float)
    agg='mean' if is_rate else 'sum'; pfx=tuple(seasons)
    if isinstance(obj,pd.Series):
        kept=[i for i in obj.index if str(i).startswith(pfx)]
        if not kept: return pd.Series(dtype=float)
        s=obj[kept]; grp=[l.split()[-1] for l in s.index]
        return s.groupby(grp).agg(agg).sort_index()
    cols=[c for c in obj.columns if str(c).startswith(pfx)]
    if not cols: return pd.DataFrame(dtype=float)
    sub=obj[cols]; grp=[c.split()[-1] for c in sub.columns]
    return sub.T.groupby(grp).agg(agg).sort_index().T

def apply_transform(s, t):
    s=pd.to_numeric(s,errors='coerce').dropna()
    if t=='YoY Growth (%)': return s.pct_change()*100
    if t=='YoY Difference': return s.diff()
    if t=='Cumulative':     return s.cumsum()
    return s

def get_raw(metric):
    return {'class_size':class_size_df,'kits_by_course':kits_course_df,'purchase_rate':rate_df,
            'giant_br':item_series.get('Giant Br',pd.Series(dtype=float)),
            'small_br':item_series.get('Small Br',pd.Series(dtype=float)),
            'wirekits':item_series.get('Wirekits',pd.Series(dtype=float)),
            'credit_card':item_series.get('Credit Card',pd.Series(dtype=float)),
            'sales':sales_series}.get(metric,pd.Series(dtype=float))

def tag(*parts): return ' · '.join(p for p in parts if p)

def build_series(seasons, metric, course_d, item_d,
                 season_mode, course_mode, item_mode, transform):
    """
    Returns dict {label: Series}.
    Each of season_mode / course_mode / item_mode is 'Aggregate' or 'Separate'.
    """
    is_rate = metric=='purchase_rate'
    if not seasons: return {}
    result = {}

    def get_for_sk(raw, sk, ir=False):
        return get_season_data(raw, sk) if sk else aggregate_by_year(raw, seasons, ir)

    season_keys = seasons if season_mode=='Separate' else [None]

    for sk in season_keys:
        slbl = sk or ''

        # ── Custom Item Aggregate ─────────────────────────────────────────────
        if metric=='item_aggregate':
            sel=get_checked(item_d)
            if not sel: continue
            if item_mode=='Separate':
                for item in sel:
                    s=get_for_sk(item_series.get(item,pd.Series(dtype=float)),sk)
                    s=apply_transform(s,transform)
                    if not s.dropna().empty: result[tag(item,slbl)]=s
            else:
                frames={i:get_for_sk(item_series.get(i,pd.Series(dtype=float)),sk) for i in sel}
                s=apply_transform(pd.DataFrame(frames).sum(axis=1),transform)
                if not s.dropna().empty: result[tag(' + '.join(sel),slbl)]=s

        # ── Course-level metrics ──────────────────────────────────────────────
        elif metric in COURSE_METRICS:
            avail=[c for c in get_checked(course_d) if c in get_raw(metric).index]
            if not avail: continue
            df=get_raw(metric).loc[avail]
            sdf=get_for_sk(df,sk,is_rate)
            if isinstance(sdf,pd.DataFrame) and sdf.empty: continue
            # ensure sdf is always a DataFrame
            if isinstance(sdf,pd.Series):
                sdf=pd.DataFrame([sdf.values],index=[avail[0]],columns=sdf.index)

            if course_mode=='Separate':
                for course in avail:
                    if course in sdf.index:
                        s=apply_transform(sdf.loc[course],transform)
                        if not s.dropna().empty: result[tag(course,slbl)]=s
            else:
                combined=sdf.mean(axis=0) if is_rate else sdf.sum(axis=0)
                lbl=avail[0] if len(avail)==1 else ' + '.join(avail)
                s=apply_transform(combined,transform)
                if not s.dropna().empty: result[tag(lbl,slbl)]=s

        # ── Single aggregate metrics (sales, giant_br, etc.) ─────────────────
        else:
            raw=get_raw(metric)
            s=get_for_sk(raw,sk)
            s=apply_transform(s,transform)
            if not s.dropna().empty:
                result[tag(METRIC_YLABEL.get(metric,metric),slbl)]=s

    return result

def project_linear(s,n=5):
    s=pd.to_numeric(s,errors='coerce').dropna()
    if len(s)<3: return [],[],[],[]
    # Trim the last year from the fit — it is likely partial
    # (e.g. 2026 only has Spring + Summer; Fall 2026 was already dropped in parse).
    # The projection line then bridges visually FROM the last complete year.
    fit_s = s.iloc[:-1] if len(s)>3 else s
    x=np.arange(len(fit_s)); y=fit_s.values.astype(float)
    sl,ic,_,_,_=spstats.linregress(x,y); sig=np.std(y-(sl*x+ic))
    last_idx=len(fit_s)-1
    # +1 so the line bridges from the last complete year into the future
    fx=np.arange(last_idx, last_idx+n+1); fy=sl*fx+ic
    last_yr=int(fit_s.index[-1])
    return [str(last_yr+i) for i in range(n+1)],fy,fy-1.96*sig,fy+1.96*sig

def series_stats(s,w):
    s=pd.to_numeric(s,errors='coerce').dropna()
    if s.empty: return None
    v=s.values.astype(float); roll=pd.Series(v).rolling(w,min_periods=1).mean()
    sl,_,r,_,_=spstats.linregress(np.arange(len(v)),v) if len(v)>1 else (0,0,0,0,0)
    return dict(mean=float(np.mean(v)),
                peak=float(np.nanmax(v)),peak_yr=str(s.index[int(np.nanargmax(v))]),
                low=float(np.nanmin(v)),low_yr=str(s.index[int(np.nanargmin(v))]),
                last_roll=float(roll.iloc[-1]),slope=sl,r2=r**2 if len(v)>1 else 0,n=len(v))

# ── Widget helpers ─────────────────────────────────────────────────────────────
def make_cbs(labels,default=True,width='103px'):
    return{l:widgets.Checkbox(value=default,description=l,indent=False,
                               layout=widgets.Layout(width=width)) for l in labels}

def sec_header(text):
    return widgets.HTML(
        f'<div style="background:#003057;color:white;padding:5px 12px;'
        f'font-weight:bold;font-size:0.88em;margin-top:4px">{text}</div>')

def hdiv():
    return widgets.HTML('<hr style="border:none;border-top:1px solid #ddd;margin:3px 0">')

def mode_btn(desc='Aggregate'):
    return widgets.ToggleButtons(
        options=['Aggregate','Separate'],value=desc,
        tooltips=['Combine all checked → one line','Each checked → its own line'],
        style={'button_width':'100px','description_width':'0px'},
        layout=widgets.Layout(margin='2px 0'))

def collapsible(title, content, open_by_default=False):
    """Reliable show/hide via a plain Button — no Tab or Accordion."""
    state={'open': open_by_default}
    btn=widgets.Button(
        description=('▼  ' if open_by_default else '▶  ')+title,
        layout=widgets.Layout(width='100%',height='32px'),
        style={'button_color':'#e8edf2','font_weight':'bold'})
    content.layout.display='' if open_by_default else 'none'
    def click(_):
        state['open']=not state['open']
        content.layout.display='' if state['open'] else 'none'
        btn.description=('▼  ' if state['open'] else '▶  ')+title
    btn.on_click(click)
    return widgets.VBox([btn,content])

# ══════════════════════════════════════════════════════════════════════════════
# WIDGETS
# ══════════════════════════════════════════════════════════════════════════════

# ── Season ────────────────────────────────────────────────────────────────────
spring_cb=widgets.Checkbox(value=True,description='Spring',indent=False,layout=widgets.Layout(width='88px'))
summer_cb=widgets.Checkbox(value=True,description='Summer',indent=False,layout=widgets.Layout(width='88px'))
fall_cb  =widgets.Checkbox(value=True,description='Fall',  indent=False,layout=widgets.Layout(width='78px'))
season_mode_tb=mode_btn('Aggregate')

# ── Metric / Transform / Chart ────────────────────────────────────────────────
metric_dd=widgets.Dropdown(options=METRIC_OPTIONS,value='class_size',
    description='Metric:',layout=widgets.Layout(width='285px'),style={'description_width':'60px'})
transform_dd=widgets.Dropdown(options=TRANSFORM_OPTIONS,value='Raw',
    description='Transform:',layout=widgets.Layout(width='235px'),style={'description_width':'82px'})
chart_toggle=widgets.ToggleButtons(options=['Line','Bar'],value='Line',
    description='Chart:',style={'button_width':'76px','description_width':'48px'})

# ── Courses ───────────────────────────────────────────────────────────────────
course_cbs=make_cbs(COURSES_CLASS,default=True,width='101px')
course_mode_tb=mode_btn('Separate')
course_area=widgets.VBox([widgets.HBox(list(course_cbs.values())),
                          widgets.HBox([widgets.HTML('<span style="color:#555;font-size:0.85em;margin-right:6px">Course Mode:</span>'),
                                        course_mode_tb])])

# ── Items ─────────────────────────────────────────────────────────────────────
item_cbs=make_cbs(ITEMS_LIST,default=False,width='130px')
item_mode_tb=mode_btn('Aggregate')
item_area=widgets.VBox([widgets.HBox(list(item_cbs.values())),
                         widgets.HBox([widgets.HTML('<span style="color:#555;font-size:0.85em;margin-right:6px">Item Mode:</span>'),
                                       item_mode_tb])],
                        layout=widgets.Layout(display='none'))

# ── Main controls (ALWAYS VISIBLE) ────────────────────────────────────────────
main_controls=widgets.VBox([
    sec_header('SEASON'),
    widgets.HBox([spring_cb,summer_cb,fall_cb],layout=widgets.Layout(margin='4px 0')),
    widgets.HBox([widgets.HTML('<span style="color:#555;font-size:0.85em;margin-right:6px">Season Mode:</span>'),
                  season_mode_tb]),
    widgets.HTML('<small style="color:#888">Aggregate: sum checked seasons per year into one line &nbsp;·&nbsp; Separate: one line per season</small>'),
    sec_header('METRIC  &amp;  DISPLAY'),
    widgets.HBox([metric_dd,transform_dd,chart_toggle],layout=widgets.Layout(gap='16px',align_items='flex-end',margin='4px 0')),
    sec_header('COURSES'),
    course_area,
    sec_header('ITEMS  (active when metric = Custom Item Aggregate)'),
    item_area,
],layout=widgets.Layout(padding='8px 14px'))

# ══════════════════════════════════════════════════════════════════════════════
# COLLAPSIBLE ADVANCED SECTIONS
# ══════════════════════════════════════════════════════════════════════════════

# ── Overlay ───────────────────────────────────────────────────────────────────
overlay_cb=widgets.Checkbox(value=False,description='Enable overlay on secondary (right) Y-axis',indent=False)
ov_metric=widgets.Dropdown(options=METRIC_OPTIONS,value='sales',
    description='Metric:',layout=widgets.Layout(width='285px'),style={'description_width':'60px'})
ov_transform=widgets.Dropdown(options=TRANSFORM_OPTIONS,value='Raw',
    description='Transform:',layout=widgets.Layout(width='235px'),style={'description_width':'82px'})
ov_course_cbs=make_cbs(COURSES_CLASS,default=False,width='101px')
ov_item_cbs  =make_cbs(ITEMS_LIST,   default=False,width='130px')
ov_season_mode_tb=mode_btn('Aggregate')
ov_course_mode_tb=mode_btn('Separate')
ov_item_mode_tb  =mode_btn('Aggregate')
ov_course_area=widgets.VBox([widgets.HBox(list(ov_course_cbs.values())),
    widgets.HBox([widgets.HTML('<span style="color:#555;font-size:0.85em;margin-right:6px">Course Mode:</span>'),ov_course_mode_tb])])
ov_item_area=widgets.VBox([widgets.HBox(list(ov_item_cbs.values())),
    widgets.HBox([widgets.HTML('<span style="color:#555;font-size:0.85em;margin-right:6px">Item Mode:</span>'),ov_item_mode_tb])],
    layout=widgets.Layout(display='none'))

overlay_content=widgets.VBox([
    overlay_cb,
    widgets.HTML('<small style="color:#888">Uses same season checkboxes. Plots on the right Y-axis.</small>'),
    widgets.HBox([ov_metric,ov_transform],layout=widgets.Layout(gap='14px',margin='4px 0')),
    widgets.HBox([widgets.HTML('<span style="color:#555;font-size:0.85em;margin-right:6px">Season Mode:</span>'),ov_season_mode_tb]),
    widgets.HTML('<b style="font-size:0.85em;color:#444">Overlay Courses:</b>'),
    ov_course_area,
    widgets.HTML('<b style="font-size:0.85em;color:#444">Overlay Items (Custom Aggregate only):</b>'),
    ov_item_area,
],layout=widgets.Layout(padding='6px 14px',border='1px solid #e0e0e0'))

# ── Rolling avg ───────────────────────────────────────────────────────────────
rolling_cb=widgets.Checkbox(value=False,description='Show rolling average line on chart',indent=False)
rolling_win=widgets.IntSlider(value=3,min=2,max=10,step=1,description='Window (yrs):',
    layout=widgets.Layout(width='360px'),style={'description_width':'100px'})
stats_html=widgets.HTML('<i style="color:#aaa">Render a chart to see statistics.</i>')

rolling_content=widgets.VBox([
    rolling_cb,rolling_win,hdiv(),
    widgets.HTML('<b style="font-size:0.85em;color:#444">Primary Series Statistics</b>'),
    stats_html,
],layout=widgets.Layout(padding='6px 14px',border='1px solid #e0e0e0'))

# ── Growth compare ────────────────────────────────────────────────────────────
ALL_YEARS=sorted({l.split()[-1] for l in class_size_df.columns})
CMP_OPTIONS=(
    [('# Total Sales','sales'),('Giant Breadboards','giant_br'),
     ('Small Breadboards','small_br'),('Wire Kits','wirekits'),('Credit Card Charges','credit_card')]
    +[(f'Class Size – {c}',('class_size',c)) for c in COURSES_CLASS]
    +[(f'Kits Sold – {c}',('kits_by_course',c)) for c in COURSES_CLASS]
    +[(f'Purchase Rate – {c}',('purchase_rate',c)) for c in ['ECE 3741','ECE 3043','ECE 2031']]
)
cmp_a=widgets.Dropdown(options=CMP_OPTIONS,value='sales',description='Series A:',
    layout=widgets.Layout(width='285px'),style={'description_width':'72px'})
cmp_b=widgets.Dropdown(options=CMP_OPTIONS,value='giant_br',description='Series B:',
    layout=widgets.Layout(width='285px'),style={'description_width':'72px'})
cmp_ys=widgets.Dropdown(options=ALL_YEARS,value=ALL_YEARS[0],description='From:',
    layout=widgets.Layout(width='165px'),style={'description_width':'44px'})
cmp_ye=widgets.Dropdown(options=ALL_YEARS,value=ALL_YEARS[-1],description='To:',
    layout=widgets.Layout(width='165px'),style={'description_width':'44px'})
cmp_btn=widgets.Button(description='Compare',button_style='primary',layout=widgets.Layout(width='110px'))
cmp_result=widgets.HTML('<i style="color:#aaa">Click Compare to see growth rate results.</i>')

compare_content=widgets.VBox([
    widgets.HTML('<small style="color:#888">CAGR = Compound Annual Growth Rate · uses same season checkboxes</small>'),
    widgets.HBox([cmp_a,cmp_b],layout=widgets.Layout(gap='12px',margin='4px 0')),
    widgets.HBox([cmp_ys,cmp_ye,cmp_btn],layout=widgets.Layout(gap='10px',align_items='flex-end')),
    hdiv(),cmp_result,
],layout=widgets.Layout(padding='6px 14px',border='1px solid #e0e0e0'))

# ── Projection ────────────────────────────────────────────────────────────────
proj_cb=widgets.Checkbox(value=False,description='Show projection on chart',indent=False)
proj_win=widgets.IntSlider(value=5,min=1,max=10,step=1,description='Horizon (yrs):',
    layout=widgets.Layout(width='360px'),style={'description_width':'100px'})
proj_html=widgets.HTML('<i style="color:#aaa">Enable the checkbox above and render to see projections.</i>')

proj_content=widgets.VBox([
    proj_cb,proj_win,
    widgets.HTML('<div style="background:#fff8e1;border-left:3px solid #ffc107;padding:6px 10px;'
                 'margin:6px 0;font-size:0.82em"><b>Note:</b> Fits a linear trend to the '
                 '<b>first</b> primary series. For GT enrollment data: '
                 '<a href="https://irp.gatech.edu" target="_blank">irp.gatech.edu</a></div>'),
    hdiv(),
    widgets.HTML('<b style="font-size:0.85em;color:#444">Projected Values</b>'),
    proj_html,
],layout=widgets.Layout(padding='6px 14px',border='1px solid #e0e0e0'))

# ══════════════════════════════════════════════════════════════════════════════
# CHART + STATS
# ══════════════════════════════════════════════════════════════════════════════
fig_widget=go.FigureWidget()
fig_widget.update_layout(
    height=430,hovermode='x unified',barmode='group',
    plot_bgcolor='white',paper_bgcolor='white',
    margin=dict(l=55,r=60,t=75,b=65),
    legend=dict(orientation='h',yanchor='bottom',y=1.02,xanchor='right',x=1,font=dict(size=10)),
    xaxis=dict(showgrid=False,tickangle=-40,title='Year'),
    yaxis=dict(gridcolor='#f0f0f0',title=''),
    yaxis2=dict(overlaying='y',side='right',showgrid=False,title='',visible=False),
    title=dict(text='Loading\u2026',font=dict(size=13,color='#003057')),
)
stats_panel=widgets.VBox(
    [widgets.HTML('<b style="color:#003057;font-size:0.88em">Series Statistics</b>'),stats_html],
    layout=widgets.Layout(width='248px',min_width='248px',border='1px solid #e0e0e0',
                          border_radius='6px',padding='10px',margin='0 0 0 10px'))

# ══════════════════════════════════════════════════════════════════════════════
# RENDER
# ══════════════════════════════════════════════════════════════════════════════
def resolve_cmp(val,seasons):
    SINGLE={'sales':sales_series,'giant_br':item_series.get('Giant Br',pd.Series(dtype=float)),
            'small_br':item_series.get('Small Br',pd.Series(dtype=float)),
            'wirekits':item_series.get('Wirekits',pd.Series(dtype=float)),
            'credit_card':item_series.get('Credit Card',pd.Series(dtype=float))}
    if isinstance(val,str): return aggregate_by_year(SINGLE[val],seasons)
    m,c=val
    df={'class_size':class_size_df,'kits_by_course':kits_course_df,'purchase_rate':rate_df}[m]
    if c not in df.index: return pd.Series(dtype=float)
    return aggregate_by_year(df.loc[[c]],seasons,m=='purchase_rate').iloc[0]

def render(_=None):
    seasons=[s for s,cb in zip(SEASON_ORDER,[spring_cb,summer_cb,fall_cb]) if cb.value]
    metric=metric_dd.value; transform=transform_dd.value; chart_t=chart_toggle.value
    sm=season_mode_tb.value; cm=course_mode_tb.value; im=item_mode_tb.value
    is_rate=metric=='purchase_rate'; n_s=len(seasons)

    # show/hide item area
    item_area.layout.display  ='' if metric=='item_aggregate' else 'none'
    ov_item_area.layout.display='' if ov_metric.value=='item_aggregate' else 'none'

    primary=build_series(seasons,metric,course_cbs,item_cbs,sm,cm,im,transform)
    overlay={}
    if overlay_cb.value:
        ov_sm=ov_season_mode_tb.value; ov_cm=ov_course_mode_tb.value; ov_im=ov_item_mode_tb.value
        raw_ov=build_series(seasons,ov_metric.value,ov_course_cbs,ov_item_cbs,
                            ov_sm,ov_cm,ov_im,ov_transform.value)
        overlay={f'[OV] {k}':v for k,v in raw_ov.items()}

    new_traces=[]
    def add_traces(sd,yax,off=0):
        for i,(lbl,s) in enumerate(sd.items()):
            s=s.dropna()
            if s.empty: continue
            clr=PALETTE[(i+off)%len(PALETTE)]; x=s.index.tolist(); y=s.values.tolist()
            ht='<b>%{fullData.name}</b><br>Year: %{x}<br>Value: %{y:,.2f}<extra></extra>'
            if chart_t=='Line':
                new_traces.append(go.Scatter(x=x,y=y,mode='lines+markers',name=lbl,
                    line=dict(color=clr,width=2.2),marker=dict(size=6),yaxis=yax,hovertemplate=ht))
                if rolling_cb.value and yax=='y':
                    w=rolling_win.value
                    rs=pd.Series(y,index=x).rolling(w,min_periods=1).mean()
                    new_traces.append(go.Scatter(
                        x=rs.index.tolist(),y=rs.values.tolist(),mode='lines',
                        name=f'{lbl} ({w}-yr avg)',line=dict(color=clr,width=2,dash='dash'),
                        opacity=0.55,yaxis=yax,
                        hovertemplate='<b>%{fullData.name}</b><br>Year: %{x}<br>Avg: %{y:,.2f}<extra></extra>'))
            else:
                new_traces.append(go.Bar(x=x,y=y,name=lbl,marker_color=clr,
                    opacity=0.82,yaxis=yax,hovertemplate=ht))

    add_traces(primary,'y',off=0)
    add_traces(overlay,'y2',off=5)

    if proj_cb.value and primary:
        fs,fv=next(iter(primary.items()))
        pyrs,py,plo,phi=project_linear(fv,proj_win.value)
        if pyrs:
            new_traces.append(go.Scatter(x=pyrs,y=py.tolist(),mode='lines+markers',
                name='Projected (linear)',line=dict(color='#e74c3c',width=2,dash='dot'),
                marker=dict(symbol='diamond',size=7,color='#e74c3c'),yaxis='y',
                hovertemplate='<b>Projection</b><br>Year: %{x}<br>Est: %{y:,.1f}<extra></extra>'))
            new_traces.append(go.Scatter(x=pyrs+pyrs[::-1],y=phi.tolist()+plo.tolist()[::-1],
                fill='toself',fillcolor='rgba(231,76,60,0.12)',
                line=dict(color='rgba(0,0,0,0)'),name='95% CI',hoverinfo='skip',yaxis='y'))
            rows=''.join(f'<tr><td style="padding:3px 6px">{yr}</td>'
                         f'<td style="text-align:right;padding:3px 6px">{v:,.0f}</td>'
                         f'<td style="text-align:right;padding:3px 6px;color:#888">'
                         f'[{lo:,.0f}\u2013{hi:,.0f}]</td></tr>'
                         for yr,v,lo,hi in zip(pyrs,py,plo,phi))
            proj_html.value=('<table style="border-collapse:collapse;width:100%;font-size:0.87em">'
                '<tr style="background:#f5f5f5"><th style="text-align:left;padding:4px 6px">Year</th>'
                '<th style="text-align:right;padding:4px 6px">Est.</th>'
                '<th style="text-align:right;padding:4px 6px">95% CI</th></tr>'+rows+'</table>')

    sm_note=(' · each season separate' if sm=='Separate' and n_s>1
             else ' (avg/yr)' if is_rate and n_s>1 else ' (sum/yr)' if n_s>1 else '')
    s_joined=', '.join(seasons) if seasons else 'none'
    title_str=f'{METRIC_YLABEL.get(metric,metric)}  \u2014  {s_joined}{sm_note}'
    ov_has=bool(overlay_cb.value and overlay)

    with fig_widget.batch_update():
        fig_widget.data=[]
        for t in new_traces: fig_widget.add_trace(t)
        fig_widget.update_layout(
            title_text=title_str, barmode='group',
            yaxis_title=METRIC_YLABEL.get(metric,''),
            # Always pass the full yaxis2 dict so overlaying/side are never silently reset
            yaxis2=dict(
                title=METRIC_YLABEL.get(ov_metric.value,'') if ov_has else '',
                visible=ov_has,
                overlaying='y',
                side='right',
                showgrid=False,
            ))

    if primary:
        fs,fv=next(iter(primary.items())); st=series_stats(fv,rolling_win.value)
        if st:
            tc='#1a7a3e' if st['slope']>0 else '#c0392b'
            ta='\u2191' if st['slope']>0 else '\u2193'
            tp=abs(st['slope']/(st['mean']+1e-9))*100; w=rolling_win.value
            stats_html.value=(
                f'<div style="font-size:0.86em;line-height:1.7">'
                f'<span style="color:#666;font-size:0.8em;word-break:break-word">{fs}</span><br>'
                f'<table style="border-collapse:collapse;width:100%;margin-top:4px">'
                f'<tr><td>Mean</td><td style="text-align:right"><b>{st["mean"]:,.1f}</b></td></tr>'
                f'<tr><td>Peak</td><td style="text-align:right"><b>{st["peak"]:,.0f}</b> '
                f'<span style="color:#888;font-size:0.8em">({st["peak_yr"]})</span></td></tr>'
                f'<tr><td>Low</td><td style="text-align:right"><b>{st["low"]:,.0f}</b> '
                f'<span style="color:#888;font-size:0.8em">({st["low_yr"]})</span></td></tr>'
                f'<tr><td>{w}-yr avg</td><td style="text-align:right"><b>{st["last_roll"]:,.1f}</b></td></tr>'
                f'<tr><td>Trend</td><td style="text-align:right">'
                f'<span style="color:{tc}"><b>{ta} {tp:.1f}%/yr</b></span></td></tr>'
                f'<tr><td>R\u00b2</td><td style="text-align:right"><b>{st["r2"]:.3f}</b></td></tr>'
                f'<tr><td>N&nbsp;(yrs)</td><td style="text-align:right"><b>{st["n"]}</b></td></tr>'
                f'</table></div>')
        else: stats_html.value='<i style="color:#aaa">No data for this selection.</i>'
    else: stats_html.value='<i style="color:#aaa">No series selected.</i>'

def do_compare(_):
    seasons=[s for s,cb in zip(SEASON_ORDER,[spring_cb,summer_cb,fall_cb]) if cb.value]
    ys,ye=cmp_ys.value,cmp_ye.value
    if ys>=ye: cmp_result.value='<span style="color:red">End year must be after start year.</span>'; return
    def gstats(s,lbl):
        s=s.dropna(); s=s[(s.index>=ys)&(s.index<=ye)]
        if len(s)<2: return lbl,None,None,None,None,None
        v0,v1=float(s.iloc[0]),float(s.iloc[-1]); n=max(len(s)-1,1)
        return lbl,v0,v1,((v1/(v0+1e-9))**(1/n)-1)*100,(v1-v0)/(abs(v0)+1e-9)*100,v1-v0
    la=next((d for d,v in CMP_OPTIONS if v==cmp_a.value),str(cmp_a.value))
    lb=next((d for d,v in CMP_OPTIONS if v==cmp_b.value),str(cmp_b.value))
    sa=gstats(resolve_cmp(cmp_a.value,seasons),la)
    sb=gstats(resolve_cmp(cmp_b.value,seasons),lb)
    def row(st):
        lbl,v0,v1,cagr,tot,absc=st
        if v0 is None: return f'<tr><td>{lbl}</td><td colspan="5" style="color:#aaa;text-align:center">No data</td></tr>'
        cc='#1a7a3e' if cagr>0 else '#c0392b'
        return(f'<tr style="border-bottom:1px solid #eee">'
               f'<td style="padding:4px 6px"><b>{lbl}</b></td>'
               f'<td style="text-align:right;padding:4px 6px">{v0:,.1f}</td>'
               f'<td style="text-align:right;padding:4px 6px">{v1:,.1f}</td>'
               f'<td style="text-align:right;padding:4px 6px;color:{cc}"><b>{cagr:+.1f}%/yr</b></td>'
               f'<td style="text-align:right;padding:4px 6px">{tot:+.1f}%</td>'
               f'<td style="text-align:right;padding:4px 6px">{absc:+,.0f}</td></tr>')
    cmp_result.value=(f'<b>Growth {ys} \u2192 {ye}</b><br><br>'
        '<table style="border-collapse:collapse;width:100%;font-size:0.87em">'
        '<tr style="background:#f5f5f5">'
        '<th style="text-align:left;padding:4px 6px">Series</th>'
        '<th style="text-align:right;padding:4px 6px">Start</th>'
        '<th style="text-align:right;padding:4px 6px">End</th>'
        '<th style="text-align:right;padding:4px 6px">CAGR</th>'
        '<th style="text-align:right;padding:4px 6px">Total \u0394</th>'
        '<th style="text-align:right;padding:4px 6px">Abs \u0394</th>'
        '</tr>'+row(sa)+row(sb)+'</table>')
cmp_btn.on_click(do_compare)

# ── Observers ─────────────────────────────────────────────────────────────────
_w=([spring_cb,summer_cb,fall_cb,season_mode_tb,metric_dd,transform_dd,chart_toggle,
     course_mode_tb,item_mode_tb,overlay_cb,ov_metric,ov_transform,
     ov_season_mode_tb,ov_course_mode_tb,ov_item_mode_tb,
     rolling_cb,rolling_win,proj_cb,proj_win]
    +list(course_cbs.values())+list(item_cbs.values())
    +list(ov_course_cbs.values())+list(ov_item_cbs.values()))
for w in _w: w.observe(render,names='value')

# ══════════════════════════════════════════════════════════════════════════════
# ASSEMBLE & DISPLAY
# ══════════════════════════════════════════════════════════════════════════════
header_html=widgets.HTML(
    '<div style="background:#003057;color:white;padding:10px 16px">'
    '<b style="font-size:1.1em">ECE Lab Kit Sales \u2014 Master Dashboard</b><br>'
    '<span style="font-size:0.8em;opacity:0.85">'
    'Hover charts for exact values \u00b7 zoom/pan on chart \u00b7 all controls reactive</span></div>')

adv_col=widgets.VBox([
    collapsible('Overlay  (secondary Y-axis)', overlay_content),
    collapsible('Rolling Average  &  Statistics', rolling_content),
    collapsible('Growth Rate Comparison', compare_content),
    collapsible('Projection', proj_content),
],layout=widgets.Layout(margin='4px 0'))

chart_row=widgets.HBox([fig_widget,stats_panel],
    layout=widgets.Layout(align_items='flex-start',width='100%'))

dashboard=widgets.VBox([
    header_html,
    main_controls,
    adv_col,
    widgets.HTML('<hr style="border:none;border-top:2px solid #003057;margin:6px 0">'),
    chart_row,
],layout=widgets.Layout(border='1px solid #ccc',border_radius='6px',width='100%'))

display(dashboard)
render()
print('\u2705 Dashboard ready.')
